<a href="https://colab.research.google.com/github/cols153/posture_checker/blob/main/notebooks/04a_pose_estimation_2D_mmpose_colab.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

⚠️ This notebook is intended to run on Google Colab (GPU recommended).

This notebook demonstrates 2D pose estimation using MMPose.  
Only (x, y) coordinates are used for visualization.  
This notebook is intended for offline demonstration and visualization only.


In [ ]:
# Check that a GPU is available in the Colab runtime
!nvidia-smi

In [ ]:
# Install CondaColab
# The runtime will restart automatically after this cell

!pip -q install -U condacolab
import condacolab
condacolab.install()

In [ ]:
# Create a dedicated Python 3.10 environment for MMPose
!conda create -y -n mmpose310 python=3.10
!conda run -n mmpose310 python -V

In [ ]:
# Install PyTorch with CUDA 12.1 inside the conda environment
!conda run -n mmpose310 python -m pip install -q -U pip setuptools wheel

!conda run -n mmpose310 python -m pip install -q -U \
torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 \
--index-url https://download.pytorch.org/whl/cu121

!conda run -n mmpose310 python -c "import torch; print('torch', torch.__version__, 'cuda?', torch.cuda.is_available())"

In [ ]:
# Install MMEngine and MMCV
!conda run -n mmpose310 python -m pip install -q -U mmengine

!conda run -n mmpose310 python -m pip install -q -U "mmcv==2.2.0" \
-f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html

!conda run -n mmpose310 python -c "import mmcv; print('mmcv', mmcv.__version__)"

In [ ]:
# Fix NumPy and OpenCV compatibility issues
!conda run -n mmpose310 python -m pip install -q -U --force-reinstall numpy==1.26.4
!conda run -n mmpose310 python -m pip uninstall -y opencv-python opencv-python-headless
!conda run -n mmpose310 python -m pip install -q opencv-python==4.10.0.84

!conda run -n mmpose310 env MPLBACKEND=Agg python -c "import numpy, cv2; print(numpy.__version__, cv2.__version__)"

In [ ]:
# Install MMDetection and extra dependencies
!conda run -n mmpose310 python -m pip install -q -U xtcocotools munkres
!conda run -n mmpose310 python -m pip install -q -U mmdet

In [ ]:
# Patch MMDetection version check to accept MMCV 2.2.0
!conda run -n mmpose310 bash -lc \
"sed -i \"s/mmcv_maximum_version = '2.2.0'/mmcv_maximum_version = '2.3.0'/\" \
/usr/local/envs/mmpose310/lib/python3.10/site-packages/mmdet/__init__.py"

In [ ]:
# Install MMPose and utility dependencies
!conda run -n mmpose310 python -m pip install -q -U --no-deps mmpose

!conda run -n mmpose310 python -m pip install -q -U \
scipy matplotlib pillow tqdm packaging pyyaml requests einops json_tricks prettytable matplotlib-inline ipython

In [ ]:
# Ensure pkg_resources is available
!conda run -n mmpose310 python -m pip install -q -U "setuptools<80"
!conda run -n mmpose310 python -c "import pkg_resources; print('pkg_resources OK')"

In [ ]:
# Check that MMPose imports correctly
!conda run -n mmpose310 env MPLBACKEND=Agg python -c "import mmpose; print('mmpose OK', mmpose.__version__)"

In [ ]:
# Clone the GitHub repository and define project paths
import os
import sys
import subprocess
from pathlib import Path

REPO_NAME = "posture_checker"
REPO_URL = "https://github.com/cols153/posture_checker.git"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_root = Path("/content") / REPO_NAME
    if not repo_root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)
    notebooks_dir = repo_root / "notebooks"
    os.chdir(notebooks_dir)
else:
    notebooks_dir = Path.cwd()

PROJECT_ROOT = notebooks_dir.parent if notebooks_dir.name == "notebooks" else notebooks_dir

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
RAW_IMAGES_DIR = RAW_DIR / "images"
RAW_VIDEOS_DIR = RAW_DIR / "videos"

PROCESSED_DIR = DATA_DIR / "processed"
KEYPOINTS_DIR = PROCESSED_DIR / "keypoints"
FEATURES_DIR = PROCESSED_DIR / "features"

LABELS_DIR = DATA_DIR / "labels"

MODELS_DIR = PROJECT_ROOT / "models"
MMPose_MODELS_DIR = MODELS_DIR / "mmpose"
MLP_MODELS_DIR = MODELS_DIR / "mlp"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
VIS_DIR = OUTPUTS_DIR / "visualizations"
PRED_DIR = OUTPUTS_DIR / "predictions"
LOGS_DIR = OUTPUTS_DIR / "logs"

ALL_DIRS = [
    DATA_DIR,
    RAW_DIR,
    RAW_IMAGES_DIR,
    RAW_VIDEOS_DIR,
    PROCESSED_DIR,
    KEYPOINTS_DIR,
    FEATURES_DIR,
    LABELS_DIR,
    MODELS_DIR,
    MMPose_MODELS_DIR,
    MLP_MODELS_DIR,
    OUTPUTS_DIR,
    VIS_DIR,
    PRED_DIR,
    LOGS_DIR,
]

for d in ALL_DIRS:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Current working directory:", Path.cwd())
print("Raw videos dir:", RAW_VIDEOS_DIR)
print("Visualizations dir:", VIS_DIR)
print("Predictions dir:", PRED_DIR)

# --- Specific paths for this notebook ---

# Input video
VIDEO_PATH = RAW_VIDEOS_DIR / "demo" / "pushup_video.mp4"

# JSON outputs
POSE2D_JSON_PATH = PROJECT_ROOT / "data" / "interim" / "keypoints_json" / "pose2d" / "pushup_video.json"
BODY3D_JSON_PATH = PROJECT_ROOT / "data" / "interim" / "keypoints_json" / "body3d" / "pushup_video.json"

# Ensure JSON folders exist
POSE2D_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
BODY3D_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)

# Output video
OUTPUT_VIDEO_PATH = OUTPUTS_DIR / "annotated_pushup_mmpose.mp4"

# Debug
print("Video exists:", VIDEO_PATH.exists())
print("2D JSON:", POSE2D_JSON_PATH)
print("3D JSON:", BODY3D_JSON_PATH)


## Project paths


In [ ]:
# Define the input demo video path inside the repository
VIDEO_IN = RAW_VIDEOS_DIR / "demo" / "pushup_video.mp4"

print("Video path:", VIDEO_IN)


In [ ]:
# Verify that the video exists before running inference
print("Video exists:", VIDEO_IN.exists())
!ls -lh "{VIDEO_IN}"


In [ ]:
## Run pose estimation on the demo push-up video

!conda run -n mmpose310 env MPLBACKEND=Agg python demo/inferencer_demo.py \
"{VIDEO_IN}" \
--pose2d human \
--device cuda \
--vis-out-dir "{VIS_DIR}" \
--pred-out-dir "{PRED_DIR}"


In [ ]:
## Run pose estimation on the push-up video

!conda run -n mmpose310 env MPLBACKEND=Agg python demo/inferencer_demo.py \
"{VIDEO_IN}" \
--pose2d human \
--device cuda \
--vis-out-dir "{VIS_DIR}" \
--pred-out-dir "{PRED_DIR}"

In [ ]:
# Check the generated output files

print("Visualization files:")
!ls -lh "{VIS_DIR}"

print("\nPrediction files:")
!ls -lh "{PRED_DIR}"